# Notebook 11: Full Model Fine-Tuning for Text Generation

## 📚 Historical Context

**Timeline**: 2018-2020 - From GPT to GPT-3

**Before Language Model Fine-Tuning (Pre-2018)**:
- **Sequence-to-sequence models**: Encoder-decoder architectures
- **Task-specific models**: Different architecture for summarization, translation, etc.
- **Limited generation**: Short, often incoherent text
- **No pre-training**: Random initialization, train from scratch
- **Data hungry**: Needed millions of examples per task

**The Breakthrough: GPT Series**

**GPT-1 (June 2018)**:
- **Pre-train**: Causal language modeling on BooksCorpus (7,000 books)
- **Fine-tune**: Add task-specific head, train on downstream task
- **Size**: 117M parameters, 12 layers
- **Innovation**: First successful decoder-only pre-training
- **Results**: SOTA on 9/12 tasks with minimal task-specific architecture

**GPT-2 (February 2019)**:
- **Scale up**: 1.5B parameters, 48 layers, 40GB of web text
- **Zero-shot**: No fine-tuning needed for many tasks
- **Controversy**: Initially withheld "due to concerns about malicious use"
- **Impact**: Showed that scale + pre-training = impressive generation
- **Limitations**: Still needed fine-tuning for specialized tasks

**GPT-3 (June 2020)**:
- **Massive scale**: 175B parameters, 96 layers
- **Few-shot learning**: No fine-tuning needed, just examples in prompt
- **End of an era**: Too large for typical fine-tuning
- **New paradigm**: Prompt engineering > fine-tuning

**Why Fine-tune Language Models (2018-2020)**:
1. **Coherent long-form text**: Better than seq2seq models
2. **Domain adaptation**: Medical reports, legal documents, code
3. **Style transfer**: Mimic specific writing styles
4. **Data efficiency**: Pre-training reduces data needs
5. **Controllability**: Fine-tuned models more controllable than prompting

**Modern Context (2023+)**:
- **Large models**: GPT-4, Llama-3, Claude use prompting, not fine-tuning
- **Small models**: GPT-2, GPT-Neo still fine-tuned for specialized tasks
- **Best of both**: Prompt engineering on base model, fine-tune for edge cases
- **LoRA/QLoRA**: When you must fine-tune large models (Stage 2)

## 🎯 What You'll Learn

1. **Causal Language Modeling** - The training objective
2. **GPT-2 Architecture** - Decoder-only transformers
3. **Perplexity** - Evaluation metric for language models
4. **Sampling Methods** - Temperature, top-k, top-p, beam search
5. **Before/After Comparison** - Measuring improvement
6. **Common Issues** - Repetition, incoherence, mode collapse

## 💡 Why This Matters

**When to fine-tune language models**:
- Domain-specific text generation (medical, legal, technical)
- Consistent style/tone (brand voice, author style)
- Edge deployment (GPT-2 fits on edge devices)
- Privacy-sensitive tasks (local inference, no API calls)
- Cost optimization (smaller fine-tuned model cheaper than large API)

**Real-world applications**:
- **Code generation**: Fine-tune on codebase for auto-completion
- **Creative writing**: Mimic author's style
- **Customer service**: Generate consistent, brand-aligned responses
- **Content creation**: Product descriptions, marketing copy
- **Data augmentation**: Generate synthetic training data

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    GPT2Config,
    get_linear_schedule_with_warmup
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Plot settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Understanding Causal Language Modeling

### What is Causal Language Modeling?

**Objective**: Given a sequence of tokens, predict the next token

**Example**:
```
Input:  "The cat sat on the"
Target: "cat sat on the mat"
```

**Training**:
- For each position i, predict token i+1
- Use causal attention mask (can't see future tokens)
- Minimize cross-entropy loss

### Causal vs Masked Language Modeling:

**Causal LM (GPT)**:
```
Input:  [The] [cat] [sat] [on] [the]
Predict: [cat] [sat] [on] [the] [mat]
```
- Predict next token
- Autoregressive: token i depends on tokens 1..i-1
- Used for generation

**Masked LM (BERT)**:
```
Input:  [The] [MASK] [sat] [on] [MASK]
Predict:      [cat]             [mat]
```
- Predict masked tokens
- Bidirectional: can see context from both sides
- Used for understanding, not generation

### Why Causal Attention?

**During training**:
- Prevents "cheating" (seeing the answer)
- Forces model to predict from left context only

**During inference**:
- Generate one token at a time
- Each token only depends on previous tokens
- Enables autoregressive generation

### Loss Calculation:

```python
# Input sequence: [1, 2, 3, 4, 5]
# Model predicts: [2, 3, 4, 5, 6]
# Loss = CrossEntropy(predictions, targets)
# Averaged over all positions
```

In [ ]:
# Load GPT-2 model and tokenizer
print("=" * 60)
print("LOADING GPT-2 MODEL")
print("=" * 60)

model_name = "gpt2"  # GPT-2 small (124M params)
# Alternatives:
# - "gpt2-medium" (355M params, better quality, slower)
# - "gpt2-large" (774M params, best quality, much slower)
# - "gpt2-xl" (1.5B params, requires 16GB+ GPU)
# - "distilgpt2" (82M params, faster, lower quality)

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 doesn't have a pad token by default
# Set pad token to eos token (common practice)
tokenizer.pad_token = tokenizer.eos_token

print(f"\nTokenizer: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"Max length: {tokenizer.model_max_length}")

# Load model
model = GPT2LMHeadModel.from_pretrained(model_name)
model = model.to(device)

# Model architecture
config = model.config
print(f"\nModel Configuration:")
print(f"  Hidden size: {config.n_embd}")
print(f"  Num layers: {config.n_layer}")
print(f"  Num attention heads: {config.n_head}")
print(f"  Vocab size: {config.vocab_size:,}")
print(f"  Context length: {config.n_positions}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Memory (FP32): {total_params * 4 / 1e9:.2f} GB")
print(f"  Memory (FP16): {total_params * 2 / 1e9:.2f} GB")

print("\n" + "=" * 60)

## Part 2: Test Pre-trained Model

### Generation Process:

1. **Start with prompt**: "Once upon a time"
2. **Encode**: Convert to token IDs
3. **Forward pass**: Get logits for next token
4. **Sample**: Choose next token based on probabilities
5. **Append**: Add chosen token to sequence
6. **Repeat**: Until max length or EOS token

### Sampling Strategies:

**Greedy decoding**:
- Always pick highest probability token
- Deterministic, fast
- Often repetitive and boring

**Temperature sampling**:
- Scale logits by temperature T
- T < 1: More confident (peaked distribution)
- T > 1: More random (flatter distribution)
- T = 1: Unchanged probabilities

**Top-k sampling**:
- Only consider top k most likely tokens
- Typical k: 40-50
- Prevents sampling unlikely tokens

**Top-p (nucleus) sampling**:
- Consider smallest set of tokens with cumulative probability >= p
- Typical p: 0.9-0.95
- Adapts to probability distribution
- Current best practice (used in GPT-3, ChatGPT)

In [ ]:
def generate_text(
    model,
    tokenizer,
    prompt,
    max_length=50,
    temperature=1.0,
    top_k=0,
    top_p=0.9,
    num_return_sequences=1
):
    """
    Generate text using various sampling strategies.
    
    Args:
        model: GPT-2 model
        tokenizer: GPT-2 tokenizer
        prompt: Input text to continue
        max_length: Maximum total length (prompt + generation)
        temperature: Sampling temperature (0.1 = conservative, 2.0 = creative)
        top_k: If > 0, only sample from top k tokens
        top_p: Nucleus sampling threshold
        num_return_sequences: Number of sequences to generate
    
    Returns:
        List of generated texts
    """
    model.eval()
    
    # Encode prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # Generate
    with torch.no_grad():
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            do_sample=True,  # Enable sampling
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode
    generated_texts = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts


# Test pre-trained model
print("=" * 60)
print("TESTING PRE-TRAINED GPT-2")
print("=" * 60)

prompt = "The future of artificial intelligence is"

print(f"\nPrompt: {prompt}\n")

# Test different sampling strategies
strategies = [
    {"name": "Greedy", "temperature": 1.0, "top_k": 1, "top_p": 1.0},
    {"name": "Low temp", "temperature": 0.5, "top_k": 0, "top_p": 0.9},
    {"name": "Medium temp", "temperature": 1.0, "top_k": 0, "top_p": 0.9},
    {"name": "High temp", "temperature": 1.5, "top_k": 0, "top_p": 0.9},
]

for strategy in strategies:
    texts = generate_text(
        model,
        tokenizer,
        prompt,
        max_length=60,
        temperature=strategy["temperature"],
        top_k=strategy["top_k"],
        top_p=strategy["top_p"],
        num_return_sequences=1
    )
    
    print(f"{strategy['name']} sampling:")
    print(f"  {texts[0]}")
    print()

print("=" * 60)

# Observations on sampling
print("\nSampling Strategy Observations:")
print("\nGreedy (deterministic):")
print("  + Always produces same output")
print("  + Fast and simple")
print("  - Often repetitive")
print("  - Lacks diversity")
print("\nLow temperature (0.5):")
print("  + More focused and coherent")
print("  + Safe, predictable")
print("  - Less creative")
print("  - Can be boring")
print("\nMedium temperature (1.0):")
print("  + Balanced creativity and coherence")
print("  + Good default choice")
print("  ~ Standard setting")
print("\nHigh temperature (1.5):")
print("  + More creative and diverse")
print("  + Surprising outputs")
print("  - Can be incoherent")
print("  - May generate nonsense")

print("\n" + "=" * 60)

## Part 3: Creating Training Dataset

### Dataset Format:

**For causal LM**:
- Just raw text
- No labels needed
- Model learns to predict next token

**Data preprocessing**:
1. **Tokenization**: Text → token IDs
2. **Chunking**: Split into fixed-length sequences
3. **Batching**: Group sequences together
4. **Attention mask**: Mark padding positions

### Context Length:

**GPT-2 max length**: 1024 tokens
- Longer than this requires special handling
- Options: truncation, sliding window, hierarchical attention

**Typical training length**: 128-512 tokens
- Balance between: context, memory, speed
- Shorter = faster training, less context
- Longer = slower training, more context

### Data Quality:

**Critical for generation**:
- Model will mimic training data style
- Garbage in → garbage out
- Clean, consistent, high-quality text essential

**Domain-specific fine-tuning**:
- Medical: PubMed abstracts, clinical notes
- Legal: Case law, contracts
- Code: GitHub repositories
- Creative: Books, stories, poems

In [ ]:
class TextGenerationDataset(Dataset):
    """
    Dataset for causal language modeling.
    
    Takes raw text, tokenizes, and creates input/target pairs.
    """
    
    def __init__(self, texts, tokenizer, max_length=128):
        """
        Args:
            texts: List of text strings
            tokenizer: GPT-2 tokenizer
            max_length: Maximum sequence length
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        
        # Tokenize and chunk all texts
        for text in texts:
            # Tokenize
            tokens = tokenizer.encode(text)
            
            # Split into chunks of max_length
            for i in range(0, len(tokens), max_length):
                chunk = tokens[i:i + max_length]
                
                # Only keep chunks that are long enough
                if len(chunk) > 10:  # Minimum length
                    self.examples.append(chunk)
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        """
        Get a single example.
        
        For causal LM:
        - Input: tokens[:-1]
        - Target: tokens[1:]
        
        Example:
        Text: "The cat sat"
        Input:  [The, cat]
        Target: [cat, sat]
        """
        tokens = self.examples[idx]
        
        # Pad if necessary
        if len(tokens) < self.max_length:
            tokens = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        
        # Convert to tensor
        tokens = torch.tensor(tokens, dtype=torch.long)
        
        # For causal LM, input and labels are the same
        # The model internally shifts them
        return {
            'input_ids': tokens,
            'labels': tokens.clone()  # Clone to create separate tensor
        }


# Create synthetic dataset
# In practice, you'd load from file or scrape from web
print("=" * 60)
print("CREATING TRAINING DATASET")
print("=" * 60)

# Synthetic story dataset (fairytale style)
# This is just for demonstration - you'd use real domain-specific data
train_texts = [
    "Once upon a time, in a magical kingdom far away, there lived a brave princess who loved adventures. She would ride her horse through the enchanted forest every morning.",
    "The old wizard lived in a tower at the edge of the village. He spent his days studying ancient spells and brewing mysterious potions.",
    "In the heart of the forest, a friendly dragon guarded a treasure of golden coins. Unlike other dragons, he loved making friends with travelers.",
    "The young knight trained every day to become stronger. He dreamed of one day defending the kingdom from evil forces.",
    "A mysterious merchant arrived in town selling magical artifacts. His shop was filled with enchanted items from distant lands.",
] * 100  # Repeat to create larger dataset

# Create dataset
max_length = 64  # Shorter for faster training in demo
train_dataset = TextGenerationDataset(train_texts, tokenizer, max_length=max_length)

print(f"\nDataset Statistics:")
print(f"  Number of text chunks: {len(train_dataset)}")
print(f"  Max sequence length: {max_length}")
print(f"  Vocabulary size: {tokenizer.vocab_size:,}")

# Create dataloader
batch_size = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

print(f"\nDataLoader:")
print(f"  Batch size: {batch_size}")
print(f"  Number of batches: {len(train_loader)}")

# Inspect a batch
batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  input_ids shape: {batch['input_ids'].shape}")
print(f"  labels shape: {batch['labels'].shape}")

# Decode a sample
sample_text = tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True)
print(f"\nSample text from batch:")
print(f"  {sample_text}")

print("\n" + "=" * 60)

## Part 4: Understanding Perplexity

### What is Perplexity?

**Definition**: Measure of how "surprised" the model is by the text

**Mathematical formula**:
```
Perplexity = exp(average cross-entropy loss)
PPL = exp(-1/N * Σ log P(token_i | context))
```

**Intuition**:
- Lower perplexity = model is confident (good)
- Higher perplexity = model is uncertain (bad)
- PPL = 10 means "on average, model is as uncertain as if choosing from 10 words"

### Example:

**Good model (low perplexity)**:
```
Context: "The cat sat on the"
Model probabilities:
  mat: 0.6
  floor: 0.3
  table: 0.1
Perplexity: ~2.5 (confident prediction)
```

**Bad model (high perplexity)**:
```
Context: "The cat sat on the"
Model probabilities (uniform across 100 words):
  Each word: 0.01
Perplexity: ~100 (very uncertain)
```

### Typical Values:

| Model | Dataset | Perplexity |
|-------|---------|------------|
| Random (uniform) | Any | Vocab size |
| N-gram model | Penn Treebank | ~150 |
| LSTM | Penn Treebank | ~80-100 |
| Transformer | Penn Treebank | ~30-40 |
| GPT-2 small | Web text | ~30 |
| GPT-2 large | Web text | ~20 |
| GPT-3 | Web text | ~15-20 |

### Why Perplexity Matters:

1. **Model comparison**: Lower is better
2. **Training progress**: Should decrease over time
3. **Overfitting detection**: Train PPL << Val PPL → overfitting
4. **Domain mismatch**: High PPL on test set → domain shift

### Limitations:

- **Not perfect**: Low PPL ≠ good generation
- **Dataset-dependent**: Can't compare across datasets
- **Ignores semantics**: May have low PPL but generate nonsense
- **Best used with**: Human evaluation of generated text

In [ ]:
def calculate_perplexity(model, dataloader, device):
    """
    Calculate perplexity on a dataset.
    
    Args:
        model: GPT-2 model
        dataloader: DataLoader with text data
        device: 'cuda' or 'cpu'
    
    Returns:
        perplexity: Float value
        avg_loss: Average cross-entropy loss
    """
    model.eval()
    
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Calculating perplexity"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            # GPT-2 model returns loss when labels are provided
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss
            
            # Accumulate loss
            # Note: loss is already averaged over the batch
            batch_size = input_ids.size(0)
            seq_len = input_ids.size(1)
            total_loss += loss.item() * batch_size * seq_len
            total_tokens += batch_size * seq_len
    
    # Average loss
    avg_loss = total_loss / total_tokens
    
    # Perplexity = exp(loss)
    perplexity = np.exp(avg_loss)
    
    return perplexity, avg_loss


# Calculate perplexity on pre-trained model
print("=" * 60)
print("BASELINE PERPLEXITY (PRE-TRAINED MODEL)")
print("=" * 60)

# Create a small validation set
val_texts = [
    "The princess discovered a hidden portal in the castle library. Through it, she could see other magical realms.",
    "An ancient prophecy spoke of a chosen hero who would save the kingdom. The young apprentice never imagined it would be him.",
] * 10

val_dataset = TextGenerationDataset(val_texts, tokenizer, max_length=max_length)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Calculate baseline perplexity
baseline_ppl, baseline_loss = calculate_perplexity(model, val_loader, device)

print(f"\nBaseline (pre-trained GPT-2):")
print(f"  Loss: {baseline_loss:.4f}")
print(f"  Perplexity: {baseline_ppl:.2f}")

print(f"\nInterpretation:")
print(f"  On average, the model is as uncertain as if choosing from {baseline_ppl:.0f} equally likely tokens")

if baseline_ppl < 50:
    print(f"  ✓ Reasonable perplexity for general pre-trained model")
elif baseline_ppl < 100:
    print(f"  ~ Moderate perplexity - model has some uncertainty")
else:
    print(f"  ⚠️ High perplexity - model struggles with this domain")

print("\n" + "=" * 60)

## Part 5: Training Loop

### Training Considerations:

**Learning rate**:
- GPT-2 fine-tuning: 1e-5 to 5e-5
- Smaller than from-scratch training
- Too high → catastrophic forgetting

**Epochs**:
- Small dataset: 3-10 epochs
- Large dataset: 1-3 epochs
- Watch for overfitting

**Gradient accumulation**:
- Effective batch size = batch_size × accumulation_steps
- Useful when GPU memory is limited
- See Notebook 16 for details

### Common Issues:

**Catastrophic forgetting**:
- Model forgets pre-trained knowledge
- Solution: Lower learning rate, fewer epochs

**Mode collapse**:
- Model generates same text repeatedly
- Solution: Lower learning rate, regularization

**Loss spikes**:
- Sudden increase in loss
- Solution: Gradient clipping, lower learning rate

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """
    Train for one epoch.
    """
    model.train()
    
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        # GPT2LMHeadModel automatically computes loss when labels provided
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        scheduler.step()
        
        # Accumulate loss
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': loss.item(),
            'ppl': np.exp(loss.item()),
            'lr': scheduler.get_last_lr()[0]
        })
    
    avg_loss = total_loss / len(dataloader)
    perplexity = np.exp(avg_loss)
    
    return avg_loss, perplexity


# Initialize training
print("=" * 60)
print("TRAINING GPT-2")
print("=" * 60)

# Hyperparameters
num_epochs = 3
learning_rate = 5e-5
weight_decay = 0.01
warmup_steps = int(0.1 * len(train_loader) * num_epochs)

print(f"\nHyperparameters:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Weight decay: {weight_decay}")
print(f"  Warmup steps: {warmup_steps}")

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# Scheduler
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Training loop
train_losses = []
train_perplexities = []
val_perplexities = []

best_val_ppl = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 60)
    
    # Train
    train_loss, train_ppl = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )
    
    # Evaluate
    val_ppl, val_loss = calculate_perplexity(model, val_loader, device)
    
    # Store metrics
    train_losses.append(train_loss)
    train_perplexities.append(train_ppl)
    val_perplexities.append(val_ppl)
    
    # Print results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val PPL:   {val_ppl:.2f}")
    
    # Save best model
    if val_ppl < best_val_ppl:
        best_val_ppl = val_ppl
        best_model_state = model.state_dict().copy()
        print(f"  ✓ New best model! (Val PPL: {val_ppl:.2f})")

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Best validation perplexity: {best_val_ppl:.2f}")
print(f"Baseline perplexity: {baseline_ppl:.2f}")
print(f"Improvement: {(baseline_ppl - best_val_ppl) / baseline_ppl * 100:.1f}%")

# Load best model
model.load_state_dict(best_model_state)
print("\nLoaded best model weights")

## Part 6: Comparing Generation Quality

### Before vs After Fine-tuning:

**What to look for**:
- **Domain adaptation**: Better understanding of domain-specific terms
- **Style matching**: Mimics training data style
- **Coherence**: More coherent continuations
- **Perplexity**: Lower perplexity on domain text

### Evaluation Methods:

**Automatic metrics**:
- Perplexity (primary metric)
- BLEU score (if you have reference texts)
- Diversity metrics (unique n-grams)

**Human evaluation**:
- Fluency: Is the text grammatical?
- Coherence: Does it make sense?
- Relevance: Does it match the prompt?
- Style: Does it match desired style?

**A/B testing**:
- Show both versions to users
- Collect preferences
- Statistical significance testing

In [ ]:
# Load pre-trained model for comparison
pretrained_model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

print("=" * 60)
print("BEFORE vs AFTER FINE-TUNING")
print("=" * 60)

# Test prompts
test_prompts = [
    "Once upon a time in a magical kingdom",
    "The brave knight embarked on a quest to",
    "In the enchanted forest, the wizard",
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"Test #{i}")
    print(f"{'='*60}")
    print(f"Prompt: {prompt}\n")
    
    # Generate with pre-trained model
    print("BEFORE (pre-trained):")
    before_texts = generate_text(
        pretrained_model,
        tokenizer,
        prompt,
        max_length=80,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=1
    )
    print(f"  {before_texts[0]}\n")
    
    # Generate with fine-tuned model
    print("AFTER (fine-tuned):")
    after_texts = generate_text(
        model,
        tokenizer,
        prompt,
        max_length=80,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=1
    )
    print(f"  {after_texts[0]}")

print("\n" + "=" * 60)
print("ANALYSIS")
print("=" * 60)

print("\nExpected improvements after fine-tuning:")
print("  1. Style: More fairytale-like language")
print("  2. Vocabulary: Uses domain-specific words (kingdom, enchanted, etc.)")
print("  3. Coherence: Better story flow")
print("  4. Perplexity: Lower on similar text")

print("\nPotential issues to watch for:")
print("  1. Overfitting: Repeating exact training phrases")
print("  2. Loss of generality: Too specific to training data")
print("  3. Mode collapse: Always generating similar text")
print("  4. Catastrophic forgetting: Lost general language ability")

print("\n" + "=" * 60)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)

# Loss curves
axes[0].plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Perplexity curves
axes[1].plot(epochs_range, train_perplexities, 'b-', label='Train PPL', linewidth=2, marker='o')
axes[1].plot(epochs_range, val_perplexities, 'r-', label='Val PPL', linewidth=2, marker='s')
axes[1].axhline(y=baseline_ppl, color='g', linestyle='--', label=f'Baseline ({baseline_ppl:.1f})', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTraining Analysis:")
print(f"  Initial train PPL: {train_perplexities[0]:.2f}")
print(f"  Final train PPL: {train_perplexities[-1]:.2f}")
print(f"  Final val PPL: {val_perplexities[-1]:.2f}")
print(f"  Baseline PPL: {baseline_ppl:.2f}")
print(f"  Improvement: {(baseline_ppl - best_val_ppl):.2f} ({(baseline_ppl - best_val_ppl) / baseline_ppl * 100:.1f}%)")

if val_perplexities[-1] > val_perplexities[0]:
    print("  ⚠️ Warning: Val perplexity increased - overfitting")
elif abs(train_perplexities[-1] - val_perplexities[-1]) / train_perplexities[-1] > 0.2:
    print("  ⚠️ Warning: Large train-val gap - possible overfitting")
else:
    print("  ✓ Training looks healthy")

## Part 7: Common Generation Pitfalls

### Problem 1: Repetition

**Symptom**: Model generates same phrase over and over
```
"The cat sat on the mat. The cat sat on the mat. The cat sat on the mat..."
```

**Causes**:
- Training data has repetitive patterns
- Greedy decoding amplifies repetition
- Model overfit to training data

**Solutions**:
- Use nucleus (top-p) sampling instead of greedy
- Add repetition penalty
- Increase temperature slightly
- Implement n-gram blocking

### Problem 2: Incoherence

**Symptom**: Text doesn't make logical sense
```
"The wizard cast a spell yesterday tomorrow will rain cats and dogs bicycle."
```

**Causes**:
- Temperature too high
- Model too small for task
- Insufficient training

**Solutions**:
- Lower temperature (0.5-0.7)
- Use nucleus sampling with p=0.9
- Train longer or use larger model
- Improve training data quality

### Problem 3: Mode Collapse

**Symptom**: All generations sound the same
```
Prompt: "The knight"
Output 1: "The knight went to the castle"
Output 2: "The knight went to the castle"
Output 3: "The knight went to the castle"
```

**Causes**:
- Overtraining on small dataset
- Learning rate too high
- Dataset lacks diversity

**Solutions**:
- Reduce learning rate
- Fewer epochs
- Increase temperature at inference
- Diversify training data

### Problem 4: Catastrophic Forgetting

**Symptom**: Model forgets general language, only knows domain
```
Prompt: "How do I cook pasta?"
Output: "The wizard cast a spell on the pasta kingdom..."
(When fine-tuned on fantasy stories)
```

**Causes**:
- Learning rate too high
- Too many epochs
- Dataset too narrow

**Solutions**:
- Use lower learning rate (1e-5 or less)
- Fewer epochs (1-3)
- Mix general text with domain text
- Consider adapter layers instead of full fine-tuning

In [ ]:
# Demonstrate common issues
print("=" * 60)
print("COMMON GENERATION PITFALLS")
print("=" * 60)

prompt = "The wizard decided to"

print(f"\nPrompt: {prompt}\n")

# Issue 1: Greedy decoding (repetitive)
print("1. GREEDY DECODING (often repetitive):")
greedy_text = generate_text(
    model, tokenizer, prompt,
    max_length=80,
    temperature=1.0,
    top_k=1,  # Greedy
    top_p=1.0,
    num_return_sequences=1
)[0]
print(f"   {greedy_text}\n")

# Issue 2: High temperature (incoherent)
print("2. HIGH TEMPERATURE (can be incoherent):")
high_temp_text = generate_text(
    model, tokenizer, prompt,
    max_length=80,
    temperature=2.0,  # Very high
    top_k=0,
    top_p=0.9,
    num_return_sequences=1
)[0]
print(f"   {high_temp_text}\n")

# Good: Nucleus sampling with moderate temperature
print("3. GOOD SETTINGS (balanced):")
good_text = generate_text(
    model, tokenizer, prompt,
    max_length=80,
    temperature=0.8,  # Moderate
    top_k=0,
    top_p=0.9,  # Nucleus sampling
    num_return_sequences=1
)[0]
print(f"   {good_text}\n")

print("=" * 60)
print("\nRECOMMENDATIONS:")
print("-" * 60)
print("\nFor most use cases:")
print("  Temperature: 0.7-0.9")
print("  Top-p: 0.9-0.95")
print("  Top-k: 0 (disabled, use top-p instead)")
print("\nFor creative writing:")
print("  Temperature: 0.9-1.2")
print("  Top-p: 0.95")
print("\nFor factual/technical:")
print("  Temperature: 0.3-0.5")
print("  Top-p: 0.9")
print("\n" + "=" * 60)

## 📊 Summary

### What We Learned:

1. **Causal Language Modeling**
   - Predict next token from left context
   - Causal attention mask prevents seeing future
   - Foundation of GPT series

2. **GPT-2 Architecture**
   - Decoder-only transformer
   - Pre-trained on web text
   - 124M to 1.5B parameters

3. **Perplexity**
   - Primary metric for language models
   - Lower = better
   - exp(cross-entropy loss)

4. **Sampling Methods**
   - Greedy: Fast but repetitive
   - Temperature: Control randomness
   - Top-k: Limit to k most likely
   - Top-p (nucleus): Current best practice

5. **Fine-tuning for Generation**
   - Domain adaptation
   - Style matching
   - Lower perplexity on domain text

6. **Common Pitfalls**
   - Repetition → use nucleus sampling
   - Incoherence → lower temperature
   - Mode collapse → less training, diverse data
   - Catastrophic forgetting → lower LR, fewer epochs

### Key Hyperparameters:

| Parameter | Training | Inference |
|-----------|----------|----------|
| Learning Rate | 1e-5 to 5e-5 | N/A |
| Epochs | 1-3 | N/A |
| Temperature | N/A | 0.7-0.9 |
| Top-p | N/A | 0.9-0.95 |
| Max Length | 128-512 | 50-200 |

### Historical Milestones:

**GPT-1 (2018)**:
- Proved decoder-only pre-training works
- 117M params, 12 layers
- Fine-tuning needed for good performance

**GPT-2 (2019)**:
- Scaled to 1.5B params
- Zero-shot capabilities emerged
- Still benefited from fine-tuning

**GPT-3 (2020)**:
- 175B params
- Few-shot learning
- End of typical fine-tuning era

**Modern (2023+)**:
- Instruction tuning > task-specific fine-tuning
- RLHF for alignment
- LoRA/QLoRA for efficient adaptation

### When to Fine-tune vs Prompt:

**Fine-tune when**:
✅ Specific domain (medical, legal, code)
✅ Consistent style required
✅ Edge deployment (latency, privacy)
✅ Cost optimization (inference cheaper than API)
✅ Small model sufficient

**Prompt when**:
✅ Using large API models (GPT-4, Claude)
✅ Task varies frequently
✅ Limited training data
✅ Need latest knowledge
✅ Quick iteration needed

### Next Notebook:
**12_training_loop_explained.ipynb**

We'll explore:
- Manual training loop from scratch (no Trainer API)
- Forward, loss, backward, optimizer step breakdown
- Gradient accumulation implementation
- Learning rate scheduling strategies
- Checkpointing and resuming training
- Logging to wandb/tensorboard
- Early stopping implementation

---

**Historical Note**: The GPT series revolutionized NLP by showing that:
1. **Scale matters**: Bigger models are qualitatively better
2. **Pre-training is key**: Self-supervised learning on massive text
3. **Decoder-only works**: Don't need encoder-decoder for most tasks
4. **Prompting > Fine-tuning**: At sufficient scale, prompting suffices

However, fine-tuning still essential for:
- Specialized domains (medical, legal, scientific)
- Edge deployment (small models, local inference)
- Cost optimization (smaller fine-tuned model vs large API)
- Privacy (no data sent to external APIs)
- Customization (exact style/format control)